# Journey (property-itinerary) migration

Tracker row **#31 (Property-itinerary)** — legacy Strapi `property_itineraries`
→ new `subcollections` (SubcollectionType **Journey** under Properties), plus
the ordered **`subcollection_postcards`** join.

Dependency chain (per tracker): **Album (#14)** gives the parent
`collection_id`, **Postcard (#16)** gives the join rows — both per-env map
files must exist (`legacy_album_id_map`, `legacy_postcard_id_map`). The seed
script already created the `journey` subcollection type.

Scope decisions (2026-08-10):
- `album` → `collection_id` (**required** in v2): via the per-env album map.
  Itineraries with **no album** or an **unmigrated album** (Designer Tours)
  are **skipped** → manual review list.
- Field map: `title`→`name`, `description`→`intro`,
  `dayWiseItinerary`→`story`, `termsAndConditions`→`tour_info`,
  `numberOfNights`→`number_of_nights`, `price`→`price`.
- `best_time_to_visits` (month rows) → `best_months` JSON array of month
  names, legacy order kept.
- `coverImage` → `cover_media_id` (new column, migration
  `20260810060000_add_subcollection_cover_and_days`) — media find-or-create
  by normalized url, same as the postcard/album migrations.
- `numberOfDays` → `number_of_days` (new column, same migration).
- `status`: `deckFreeze` / `onTrip` / `complete` → `live`;
  `deckBuild` / `draft` / empty → `draft` (counts printed for review).
- `managed_by_company_id` inherited from the parent collection.
- `slug`: from legacy slug else title, de-duplicated in-run (id-sorted, so
  suffixes stay stable across re-runs).
- `postcards` (m2m) → `subcollection_postcards`, `sequence_order` = position
  in the legacy relation order. Postcards not in the postcard map (Designer
  Tours skips) are flagged; postcards whose `collection_id` differs from the
  journey's collection violate the schema invariant → **skipped** + flagged.
- **Dropped (review before running):**
  - `priceType` (`per person` / `twin sharing`) — no v2 home; v2 `price` is
    documented as avg-price-per-person. Twin-sharing rows are listed for
    manual review.
  - `country` — subcollection has no geo; inherits from parent collection.
  - `createdAt`/`updatedAt`/`publishedAt` — no timestamps on subcollections.
- `price_starting_at`, `guests_min`/`guests_max` stay NULL — no legacy source.
- `createdByUser` → OPTIONAL author circle (`owned_type='subcollection'`),
  section 6.
- Writes `legacy_itinerary_id_map_dev/_prod.json` (legacy itinerary id →
  new subcollection id) for the future Enquiry/Circle migrations.

Run cells top to bottom. Idempotent — upsert on `slug`, join upserts on its
PK. Safe to re-run.

In [1]:
import os, re, json
from pathlib import Path

import requests
import psycopg
from psycopg.types.json import Json
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]
ENV_SUFFIX = {"development": "_dev", "production": "_prod"}.get(DATABASE_URL.rsplit("/", 1)[-1], "")


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", (text or "").lower()).strip("-") or None


def attrs(item):
    """Entry fields — Strapi v4 nests them under 'attributes', v5 is flat."""
    return item.get("attributes", item)


def rel(obj):
    """Unwrap a populated relation — v4: {'data': {'attributes': {...}}}, v5: flat dict."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    if not obj:
        return None
    return obj.get("attributes", obj)


def rel_many(obj):
    """Unwrap a populated to-many relation into a list of flat dicts."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    return [attrs(x) for x in (obj or [])]


def fetch_all(path, params=None):
    """Fetch every page of a Strapi collection endpoint (data/meta envelope)."""
    items, page = [], 1
    while True:
        p = {"pagination[page]": page, "pagination[pageSize]": 100, "sort": "id", **(params or {})}
        r = requests.get(f"{CMS_BASE_URL}{path}", headers=HEADERS, params=p, timeout=120)
        r.raise_for_status()
        body = r.json()
        items.extend(body["data"])
        pg = body.get("meta", {}).get("pagination", {})
        if page >= pg.get("pageCount", 1):
            return items
        page += 1


conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])

# per-environment maps from the album and postcard migrations
album_map = {int(k): int(v) for k, v in
             json.loads((ROOT / f"legacy_album_id_map{ENV_SUFFIX}.json").read_text()).items()}
postcard_map = {int(k): int(v) for k, v in
                json.loads((ROOT / f"legacy_postcard_id_map{ENV_SUFFIX}.json").read_text()).items()}
print(f"loaded {len(album_map)} album mappings, {len(postcard_map)} postcard mappings ({ENV_SUFFIX or 'no suffix'})")

connected to: development
loaded 1261 album mappings, 2422 postcard mappings (_dev)


## 1. Fetch all property-itineraries

`populate=*` brings one level: `album`, `postcards` (ids + legacy relation
order), `best_time_to_visits` (month rows), `country`, `createdByUser`.

In [2]:
itineraries = sorted(fetch_all("/api/property-itineraries", {"populate": "*"}), key=lambda x: x["id"])
print(f"fetched {len(itineraries)} property-itineraries")

# quick shape check on the first entry
if itineraries:
    a = attrs(itineraries[0])
    print({k: type(v).__name__ for k, v in a.items()})

fetched 38 property-itineraries
{'id': 'int', 'title': 'str', 'description': 'str', 'numberOfDays': 'int', 'numberOfNights': 'int', 'price': 'int', 'status': 'str', 'createdAt': 'str', 'updatedAt': 'str', 'dayWiseItinerary': 'str', 'termsAndConditions': 'str', 'priceType': 'NoneType', 'slug': 'str', 'coverImage': 'dict', 'postcards': 'list', 'album': 'dict', 'best_time_to_visits': 'list', 'country': 'NoneType', 'createdByUser': 'NoneType'}


## 2. DB lookups

Journey subcollection-type id (seeded), and per-collection company for the
`managed_by_company_id` inheritance.

In [3]:
conn.rollback()  # clear any aborted transaction from a previous failed run

with conn.cursor() as cur:
    cur.execute("SELECT id FROM subcollection_types WHERE slug = 'journey'")
    row = cur.fetchone()
    assert row, "subcollection_types has no 'journey' row — run scripts/seed.py first"
    JOURNEY_TYPE_ID = row[0]

    cur.execute("SELECT id, managed_by_company_id FROM collections")
    company_by_collection = dict(cur.fetchall())

    cur.execute("SELECT url, id FROM media")
    media_by_url = dict(cur.fetchall())

print(f"journey subcollection_type id: {JOURNEY_TYPE_ID}")
print(f"lookups: {len(company_by_collection)} collections, {len(media_by_url)} media")


def media_id_for(image, cur):
    """Find-or-create a media row for a populated Strapi file (keyed by
    normalized url, same as scripts/media.py — rows are reused, never duplicated)."""
    if not image or not image.get("url"):
        return None
    url = image["url"].strip()
    if url.startswith("/"):
        url = CMS_BASE_URL + url
    if url in media_by_url:
        return media_by_url[url]
    cur.execute(
        "INSERT INTO media (url, mime_type, alt, width, height) VALUES (%s, %s, %s, %s, %s) RETURNING id",
        (url, image.get("mime"), image.get("alternativeText") or image.get("name"),
         image.get("width"), image.get("height")),
    )
    media_by_url[url] = cur.fetchone()[0]
    return media_by_url[url]


journey subcollection_type id: 1
lookups: 1261 collections, 9743 media


## 3. property-itinerary → `subcollections`

Upsert on `slug`. Skips: no title, no album, album not in the map
(= Designer Tours → dx-card migration later).

In [4]:
conn.rollback()

used_slugs = set()
def unique_slug(base):
    base = base or "journey"
    slug, n = base, 2
    while slug in used_slugs:
        slug = f"{base}-{n}"
        n += 1
    used_slugs.add(slug)
    return slug

# deckFreeze/onTrip/complete -> live; deckBuild/draft/None -> draft
STATUS_MAP = {"deckFreeze": "live", "onTrip": "live", "complete": "live"}

itinerary_map = {}         # legacy itinerary id -> new subcollection id
collection_of = {}         # legacy itinerary id -> new collection id (for the join invariant)

skipped_no_title, skipped_no_album, skipped_unmigrated_album = [], [], []
twin_sharing = []          # dropped priceType != 'per person' -> manual review
status_counts = {}

with conn.cursor() as cur:
    for it in itineraries:
        a = attrs(it)
        title = (a.get("title") or "").strip()
        if not title:
            skipped_no_title.append(it["id"])
            continue

        # album -> parent collection (required in v2)
        album = rel(a.get("album"))
        if not album:
            skipped_no_album.append((it["id"], title))
            continue
        collection_id = album_map.get(album["id"])
        if not collection_id:  # Designer Tours album -> dx-card migration later
            skipped_unmigrated_album.append((it["id"], title, album.get("name")))
            continue

        # best_time_to_visits month rows -> best_months JSON (legacy order kept)
        months = [m.get("name") for m in rel_many(a.get("best_time_to_visits")) if m.get("name")]

        legacy_status = a.get("status")
        status_counts[legacy_status] = status_counts.get(legacy_status, 0) + 1
        status = STATUS_MAP.get(legacy_status, "draft")

        if (a.get("priceType") or "per person") != "per person":
            twin_sharing.append((it["id"], title, a.get("priceType"), a.get("price")))

        slug = unique_slug((a.get("slug") or "").strip() or slugify(title))
        cover_id = media_id_for(rel(a.get("coverImage")), cur)

        cur.execute(
            """
            INSERT INTO subcollections
                (subcollection_type_id, collection_id, name, intro, story, slug,
                 tour_info, price, price_starting_at, number_of_nights, number_of_days,
                 cover_media_id, guests_min, guests_max, best_months,
                 managed_by_company_id, status)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, NULL, %s, %s, %s, NULL, NULL, %s, %s, %s)
            ON CONFLICT (slug) DO UPDATE
            SET subcollection_type_id = EXCLUDED.subcollection_type_id,
                collection_id = EXCLUDED.collection_id,
                name = EXCLUDED.name,
                intro = EXCLUDED.intro,
                story = EXCLUDED.story,
                tour_info = EXCLUDED.tour_info,
                price = EXCLUDED.price,
                number_of_nights = EXCLUDED.number_of_nights,
                number_of_days = EXCLUDED.number_of_days,
                cover_media_id = EXCLUDED.cover_media_id,
                best_months = EXCLUDED.best_months,
                managed_by_company_id = EXCLUDED.managed_by_company_id,
                status = EXCLUDED.status
            RETURNING id
            """,
            (JOURNEY_TYPE_ID, collection_id, title,
             (a.get("description") or "").strip() or None,
             (a.get("dayWiseItinerary") or "").strip() or None,
             slug,
             (a.get("termsAndConditions") or "").strip() or None,
             a.get("price"),
             a.get("numberOfNights"),
             a.get("numberOfDays"),
             cover_id,
             Json(months) if months else None,
             company_by_collection.get(collection_id),
             status),
        )
        itinerary_map[it["id"]] = cur.fetchone()[0]
        collection_of[it["id"]] = collection_id

conn.commit()
print(f"subcollections upserted: {len(itinerary_map)}")
print(f"legacy status counts (deckFreeze/onTrip/complete -> live): {status_counts}")
print(f"skipped (no title): {skipped_no_title}")
print(f"skipped (no album — journey needs a parent Property) ({len(skipped_no_album)}): {skipped_no_album}")
print(f"skipped, album not migrated = Designer Tours ({len(skipped_unmigrated_album)}): {skipped_unmigrated_album}")
print(f"MANUAL REVIEW priceType != 'per person' (dropped field) ({len(twin_sharing)}): {twin_sharing}")


subcollections upserted: 29
legacy status counts (deckFreeze/onTrip/complete -> live): {'deckFreeze': 5, None: 4, 'draft': 10, 'deckBuild': 10}
skipped (no title): []
skipped (no album — journey needs a parent Property) (0): []
skipped, album not migrated = Designer Tours (9): [(31, 'Hyderabad in Three Days', 'AdZENtures'), (32, 'newwww Hyderabad in Three Days', 'A Space Inspired Adventure in Otherworldly Antarctica'), (33, 'Hyderabad in Three Days', 'A 15-day Transformational Journey in Inspiring Sri Lanka'), (34, '5 Nights at Evolve Back Coorg', 'A 15-day Transformational Journey in Inspiring Sri Lanka'), (35, 'Udaipur to Jalore: Lakes, Marble & Granite Forts', 'A 15-day Transformational Journey in Inspiring Sri Lanka'), (36, "Udaipur, Mt. Abu & Jalore: Rajasthan's Western Arc", 'Ace The Himalaya'), (37, '5 Nights at Evolve Back Coorg', 'Ace The Himalaya'), (39, 'Hyderabad in Three Days', 'A Space Inspired Adventure in Otherworldly Antarctica'), (40, '5 Nights at Evolve Back Coorg', 

## 4. postcards → `subcollection_postcards`

`sequence_order` = position in the legacy m2m order (Day 1, Day 2, ...).
Schema invariant: the postcard's `collection_id` must equal the journey's
`collection_id` — violating links are **skipped** and listed for manual
review. Upsert on the (subcollection, postcard) PK so re-runs refresh the
order.

In [ ]:
conn.rollback()

with conn.cursor() as cur:
    cur.execute("SELECT id, collection_id FROM postcards")
    postcard_collection = dict(cur.fetchall())

links = 0
unmapped_postcards, cross_property = [], []

with conn.cursor() as cur:
    for it in itineraries:
        sub_id = itinerary_map.get(it["id"])
        if not sub_id:
            continue
        coll_id = collection_of[it["id"]]
        order = 0
        for p in rel_many(attrs(it).get("postcards")):
            new_pid = postcard_map.get(p["id"])
            if not new_pid:  # postcard skipped in #16 (Designer Tours)
                unmapped_postcards.append((it["id"], p["id"], p.get("name")))
                continue
            if postcard_collection.get(new_pid) != coll_id:
                cross_property.append((it["id"], p["id"], p.get("name")))
                continue
            order += 1
            cur.execute(
                """
                INSERT INTO subcollection_postcards (subcollection_id, postcard_id, sequence_order)
                VALUES (%s, %s, %s)
                ON CONFLICT (subcollection_id, postcard_id) DO UPDATE
                SET sequence_order = EXCLUDED.sequence_order
                """,
                (sub_id, new_pid, order),
            )
            links += 1

conn.commit()
print(f"subcollection_postcards upserted: {links}")
print(f"MANUAL REVIEW postcard not in map ({len(unmapped_postcards)}): {unmapped_postcards[:20]}")
print(f"MANUAL REVIEW postcard belongs to a different collection than the journey "
      f"(invariant — skipped) ({len(cross_property)}): {cross_property[:20]}")

## 5. Save the legacy itinerary id map

`legacy_itinerary_id_map_dev/_prod.json` (legacy property-itinerary id → new
subcollection id) — future Enquiry (#27) / Circle bookings migrations need it.

In [ ]:
out = ROOT / f"legacy_itinerary_id_map{ENV_SUFFIX}.json"
out.write_text(json.dumps({str(k): str(v) for k, v in itinerary_map.items()}, indent=2))
print(f"saved {len(itinerary_map)} legacy->new itinerary id mappings to {out}")

## 6. OPTIONAL — author circles

Legacy `createdByUser` → Circle `author` (`owned_type = 'subcollection'`),
via the per-env legacy user id map. Skip this cell if circles should wait.

In [ ]:
conn.rollback()

user_map_file = ROOT / f"legacy_user_id_map{ENV_SUFFIX}.json"
user_map = {int(k): int(v) for k, v in json.loads(user_map_file.read_text()).items()}
print(f"loaded {len(user_map)} user mappings from {user_map_file.name}")

author_rows = 0
unmapped_users = []
with conn.cursor() as cur:
    for it in itineraries:
        sub_id = itinerary_map.get(it["id"])
        if not sub_id:
            continue
        u = rel(attrs(it).get("createdByUser"))
        if not u:
            continue
        new_uid = user_map.get(u["id"])
        if not new_uid:
            unmapped_users.append((it["id"], u["id"]))
            continue
        cur.execute(
            """
            INSERT INTO circles (user_id, owned_type, owned_id, relationship)
            VALUES (%s, 'subcollection', %s, 'author')
            ON CONFLICT (user_id, owned_type, owned_id, relationship) DO NOTHING
            """,
            (new_uid, sub_id),
        )
        author_rows += cur.rowcount

conn.commit()
print(f"author circles inserted this run: {author_rows}")
print(f"MANUAL REVIEW legacy users not in map ({len(unmapped_users)}): {unmapped_users[:20]}")

## 7. Verification

Expected: every journey has a parent collection (FK-enforced), join rows ≈
sum of legacy itinerary-postcard links minus the two skip lists, 0 duplicate
slugs, 0 invariant violations.

In [ ]:
with conn.cursor() as cur:
    for label, q in [
        ("subcollections total",  "SELECT COUNT(*) FROM subcollections"),
        ("journeys",              "SELECT COUNT(*) FROM subcollections s JOIN subcollection_types t ON t.id = s.subcollection_type_id WHERE t.slug = 'journey'"),
        ("with price",            "SELECT COUNT(*) FROM subcollections WHERE price IS NOT NULL"),
        ("with nights",           "SELECT COUNT(*) FROM subcollections WHERE number_of_nights IS NOT NULL"),
        ("with days",             "SELECT COUNT(*) FROM subcollections WHERE number_of_days IS NOT NULL"),
        ("with cover media",      "SELECT COUNT(*) FROM subcollections WHERE cover_media_id IS NOT NULL"),
        ("with best_months",      "SELECT COUNT(*) FROM subcollections WHERE best_months IS NOT NULL"),
        ("with company",          "SELECT COUNT(*) FROM subcollections WHERE managed_by_company_id IS NOT NULL"),
        ("status = live",         "SELECT COUNT(*) FROM subcollections WHERE status = 'live'"),
        ("join rows",             "SELECT COUNT(*) FROM subcollection_postcards"),
        ("journeys w/ postcards", "SELECT COUNT(DISTINCT subcollection_id) FROM subcollection_postcards"),
        ("empty journeys",        "SELECT COUNT(*) FROM subcollections s WHERE NOT EXISTS (SELECT 1 FROM subcollection_postcards sp WHERE sp.subcollection_id = s.id)"),
        ("author circles",        "SELECT COUNT(*) FROM circles WHERE owned_type = 'subcollection' AND relationship = 'author'"),
        ("dup slugs (want 0)",    "SELECT COUNT(*) FROM (SELECT slug FROM subcollections GROUP BY slug HAVING COUNT(*) > 1) d"),
        ("invariant viol. (want 0)", "SELECT COUNT(*) FROM subcollection_postcards sp JOIN subcollections s ON s.id = sp.subcollection_id JOIN postcards p ON p.id = sp.postcard_id WHERE p.collection_id IS DISTINCT FROM s.collection_id"),
    ]:
        cur.execute(q)
        print(f"{label:26}: {cur.fetchone()[0]}")

    # per-property journey counts (top 15)
    cur.execute("""
        SELECT c.name, COUNT(s.id) AS journeys
        FROM subcollections s JOIN collections c ON c.id = s.collection_id
        GROUP BY c.id, c.name ORDER BY journeys DESC, c.name LIMIT 15
    """)
    print("\ntop properties by journey count:")
    for name, n in cur.fetchall():
        print(f"  {name:40}: {n}")
conn.close()